<a href="https://colab.research.google.com/github/cybercolombia/suelosabio/blob/feature%2FSCRUM-13/notebooks/ClimatePipeline/06_ClimateGeographyAudit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ClimateGeographyAudit

Construye y audita un catálogo preliminar `estación → municipio` a partir del clima diario curado, el catálogo IDEAM y DIVIPOLA.

## Alcance de esta versión

- Lee `eco2026` como fuente compartida de **solo lectura**.
- Escribe únicamente en `eco2026_processed/geografia_curada`.
- Genera un mapa interactivo de puntos sobre OpenStreetMap.
- Conserva municipio y coordenadas observadas, catálogo IDEAM y candidato DIVIPOLA.
- Marca discrepancias de departamento, municipio y coordenadas.
- No declara asignaciones canónicas mientras falte una capa municipal completa para validar punto-en-polígono.

Este notebook no agrega clima por municipio y no imputa estaciones.

## 1. Preparar el repositorio

La celda clona o actualiza la rama para importar exactamente las reglas geográficas versionadas.

In [ ]:
from pathlib import Path

import subprocess
import sys

try:
    from google.colab import drive

    drive.mount('/content/drive')
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

REPO_URL = 'https://github.com/cybercolombia/suelosabio.git'
REPO_REF = 'feature/SCRUM-13'
REPO_DIR = Path('/content/suelosabio') if IN_COLAB else Path.cwd()
ACTUALIZAR_REPOSITORIO = True

if IN_COLAB:
    if not (REPO_DIR / '.git').exists():
        subprocess.run(
            ['git', 'clone', '--depth', '1', '--branch', REPO_REF, REPO_URL, str(REPO_DIR)],
            check=True,
        )
    elif ACTUALIZAR_REPOSITORIO:
        subprocess.run(['git', 'fetch', 'origin', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'checkout', REPO_REF], cwd=REPO_DIR, check=True)
        subprocess.run(['git', 'pull', '--ff-only', 'origin', REPO_REF], cwd=REPO_DIR, check=True)

PIPELINE_DIR = REPO_DIR / 'notebooks' / 'ClimatePipeline'
if not PIPELINE_DIR.exists():
    raise FileNotFoundError(f'No existe la carpeta del pipeline: {PIPELINE_DIR}')
if str(PIPELINE_DIR) not in sys.path:
    sys.path.insert(0, str(PIPELINE_DIR))

print({'in_colab': IN_COLAB, 'repo_ref': REPO_REF, 'repo_dir': str(REPO_DIR)})

## 2. Configuración protegida

La carpeta compartida contiene únicamente entradas. La salida se dirige a la carpeta personal procesada.

In [ ]:
import json
import time

import pandas as pd

from ClimateGeography import GEOGRAPHY_VERSION, auditar_geografia
from ClimateProcessingUtils import (
    ahora_proyecto,
    detectar_commit,
    escribir_json_atomico,
    escribir_parquet_atomico,
    escribir_texto_atomico,
    formatear_duracion,
    slugificar,
)

try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = str

    def display(valor):
        print(valor)

VARIABLE_NOMBRE = 'precipitacion'
DATASET_ID = 's54a-sgyg'
CONSOLIDACION_NOMBRE = 'cierre_precipitacion_2024_2025_v2'
CONSOLIDATION_VERSION_ESPERADA = 'precipitacion_estacion_dia_v2'
EJECUCION_GEOGRAFICA = 'estaciones_precipitacion_2024_2025_v1'
UMBRAL_COORDENADAS_GRADOS = 0.001

EJECUTAR_AUDITORIA_GEOGRAFICA = False
GUARDAR_RESULTADOS = True
SOBRESCRIBIR_RESULTADOS = False

SHARED_SOURCE_ROOT = (
    Path('/content/drive/MyDrive/eco2026')
    if IN_COLAB
    else Path.cwd() / 'local_docs' / 'tmp' / 'geografia'
)
PROCESSED_ROOT = (
    Path('/content/drive/MyDrive/eco2026_processed')
    if IN_COLAB
    else Path.cwd() / 'eco2026_processed'
)
ESTACIONES_PATH = SHARED_SOURCE_ROOT / 'Estaciones_IDEAM_20260527.csv'
DIVIPOLA_CSV_PATH = SHARED_SOURCE_ROOT / 'Divipola.csv'
DIVIPOLA_JSON_PATH = SHARED_SOURCE_ROOT / 'Divipola_Municipios.json'
CLIMATE_INPUT_DIR = (
    PROCESSED_ROOT
    / 'clima_diario_curado'
    / f'variable={VARIABLE_NOMBRE}'
    / f'fuente={DATASET_ID}'
    / f'consolidacion={slugificar(CONSOLIDACION_NOMBRE)}'
)
OUTPUT_DIR = (
    PROCESSED_ROOT
    / 'geografia_curada'
    / f'ejecucion={slugificar(EJECUCION_GEOGRAFICA)}'
)

if OUTPUT_DIR == SHARED_SOURCE_ROOT or SHARED_SOURCE_ROOT in OUTPUT_DIR.parents:
    raise RuntimeError('La salida geográfica no puede escribirse dentro de eco2026.')

print({
    'geography_version': GEOGRAPHY_VERSION,
    'ejecutar': EJECUTAR_AUDITORIA_GEOGRAFICA,
    'guardar': GUARDAR_RESULTADOS,
    'fuente_solo_lectura': str(SHARED_SOURCE_ROOT),
    'entrada_clima': str(CLIMATE_INPUT_DIR),
    'salida_propia': str(OUTPUT_DIR),
})

In [ ]:
def inspeccionar_plan_geografico():
    manifest_path = CLIMATE_INPUT_DIR / 'manifest.json'
    manifest = {}
    if manifest_path.exists():
        manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    archivos = sorted(
        CLIMATE_INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/observaciones_estacion_dia.parquet'
        )
    )
    return pd.DataFrame([{
        'entrada_clima_estado': manifest.get('estado', 'NO_ENCONTRADA'),
        'regla_clima': manifest.get('regla_version'),
        'particiones_clima': len(archivos),
        'catalogo_ideam': ESTACIONES_PATH.exists(),
        'divipola_csv': DIVIPOLA_CSV_PATH.exists(),
        'divipola_json': DIVIPOLA_JSON_PATH.exists(),
        'salida_fuera_de_compartida': SHARED_SOURCE_ROOT not in OUTPUT_DIR.parents,
        'salida': str(OUTPUT_DIR),
    }])


plan_geografico_df = inspeccionar_plan_geografico()
display(Markdown('### Plan de auditoría geográfica'))
display(plan_geografico_df)

## 3. Lectura, mapa y escritura segura

La ejecución exige un manifiesto climático `COMPLETA`, 48 particiones y las dos fuentes DIVIPOLA concordantes para Boyacá y Cundinamarca.

In [ ]:
NOMBRES_SALIDA = {
    'catalogo': 'catalogo_estaciones_climaticas.parquet',
    'candidatos': 'estaciones_municipio_candidato.parquet',
    'revision': 'estaciones_revision.parquet',
    'divipola': 'divipola_municipios.parquet',
    'resumen': 'resumen_geografico.parquet',
    'mapa': 'mapa_estaciones.html',
    'reporte': 'AuditoriaGeografica_estaciones_precipitacion_2024_2025.md',
    'manifest': 'manifest.json',
}


def validar_divipolas_concordantes(divipola_csv, json_path):
    contenido = json.loads(json_path.read_text(encoding='utf-8'))
    divipola_json = pd.DataFrame(contenido['resultado'])
    codigos_csv = set(
        divipola_csv.loc[
            divipola_csv['Código Departamento'].astype('string').str.zfill(2).isin(['15', '25']),
            'Código Municipio',
        ].astype('string').str.zfill(5)
    )
    codigos_json = set(
        divipola_json.loc[
            divipola_json['CODIGO_DEPARTAMENTO'].astype('string').str.zfill(2).isin(['15', '25']),
            'CODIGO_DPTO_MPIO',
        ].astype('string').str.zfill(5)
    )
    if codigos_csv != codigos_json:
        raise RuntimeError('Divipola.csv y Divipola_Municipios.json no concuerdan en el alcance.')
    if len(codigos_csv) != 239:
        raise RuntimeError(f'Se esperaban 239 municipios objetivo y se encontraron {len(codigos_csv)}.')


def cargar_entradas_geograficas():
    manifest_path = CLIMATE_INPUT_DIR / 'manifest.json'
    if not manifest_path.exists():
        raise FileNotFoundError(f'No existe el manifiesto climático: {manifest_path}')
    manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
    if manifest.get('estado') != 'COMPLETA':
        raise RuntimeError('La consolidación climática de entrada no está COMPLETA.')
    if manifest.get('regla_version') != CONSOLIDATION_VERSION_ESPERADA:
        raise RuntimeError(
            f"Versión climática inesperada: {manifest.get('regla_version')}"
        )
    archivos = sorted(
        CLIMATE_INPUT_DIR.glob(
            'departamento=*/anio=*/mes=*/observaciones_estacion_dia.parquet'
        )
    )
    if len(archivos) != 48:
        raise RuntimeError(f'Se esperaban 48 particiones climáticas y existen {len(archivos)}.')
    diario = pd.concat([pd.read_parquet(archivo) for archivo in archivos], ignore_index=True)
    filas_manifest = manifest.get('metricas', {}).get('filas_estacion_dia_salida')
    if filas_manifest is not None and len(diario) != int(filas_manifest):
        raise RuntimeError(f'Filas climáticas ({len(diario):,}) != manifiesto ({filas_manifest:,}).')
    estaciones = pd.read_csv(ESTACIONES_PATH, dtype='string')
    divipola = pd.read_csv(DIVIPOLA_CSV_PATH, dtype='string')
    validar_divipolas_concordantes(divipola, DIVIPOLA_JSON_PATH)
    return diario, estaciones, divipola, manifest, archivos


def tabla_markdown(tabla, limite=None):
    vista = tabla.head(limite) if limite is not None else tabla
    try:
        return vista.to_markdown(index=False)
    except ImportError:
        return '```text\n' + vista.to_string(index=False) + '\n```'


def construir_reporte(resultado, manifest_clima, inicio, fin, duracion):
    return '\n'.join([
        '# Auditoría geográfica de estaciones climáticas',
        '',
        f'- Contrato geográfico: `{GEOGRAPHY_VERSION}`',
        f'- Consolidación climática: `{manifest_clima.get("commit")}`',
        f'- Commit auditor: `{detectar_commit(REPO_DIR)}`',
        f'- Estado: `{resultado.metricas["estado"]}`',
        f'- Inicio: `{inicio.isoformat()}`',
        f'- Fin: `{fin.isoformat()}`',
        f'- Duración: `{formatear_duracion(duracion)}`',
        '',
        '> Las asignaciones son candidatas de catálogo. Ninguna es canónica hasta validar punto-en-polígono.',
        '',
        '## Métricas',
        '',
        tabla_markdown(pd.DataFrame([resultado.metricas])),
        '',
        '## Resumen por departamento de descarga',
        '',
        tabla_markdown(resultado.resumen),
        '',
        '## Estaciones para revisión',
        '',
        tabla_markdown(
            resultado.estaciones_revision[[
                'departamento', 'codigoestacion', 'Nombre', 'Municipio',
                'municipios_reportados', 'codigo_municipio',
                'motivos_revision_geografica',
            ]],
        ),
        '',
        '## Limitación',
        '',
        'La capa `Div_Pol.shp` disponible no incluye todos los componentes del shapefile. '
        'Esta ejecución no realiza punto-en-polígono ni produce un catálogo canónico.',
        '',
    ])


def construir_mapa(resultado):
    try:
        import plotly.express as px
    except ImportError:
        print('⚠️ Plotly no está disponible; se omite el mapa, pero continúa la auditoría tabular.')
        return None
    tabla = resultado.estaciones_candidatas.copy()
    tabla['estado_mapa'] = tabla['requiere_revision_geografica'].map({
        False: 'Candidato catálogo OK',
        True: 'Requiere revisión',
    })
    argumentos = dict(
        data_frame=tabla,
        lat='LATITUD',
        lon='LONGITUD',
        color='estado_mapa',
        color_discrete_map={
            'Candidato catálogo OK': '#287271',
            'Requiere revisión': '#d1662f',
        },
        hover_name='Nombre',
        hover_data={
            'codigoestacion': True,
            'Departamento': True,
            'Municipio': True,
            'codigo_municipio': True,
            'altitud_ideam_m': True,
            'Estado': True,
            'motivos_revision_geografica': True,
            'LATITUD': ':.5f',
            'LONGITUD': ':.5f',
        },
        center={'lat': 5.35, 'lon': -73.85},
        zoom=6,
        height=720,
        title='Estaciones climáticas y candidatos geográficos 2024-2025',
    )
    if hasattr(px, 'scatter_map'):
        figura = px.scatter_map(**argumentos, map_style='open-street-map')
    else:
        figura = px.scatter_mapbox(**argumentos, mapbox_style='open-street-map')
    figura.update_traces(marker={'size': 10, 'opacity': 0.85})
    figura.update_layout(margin={'l': 0, 'r': 0, 't': 55, 'b': 0})
    return figura


def guardar_auditoria_geografica(resultado, reporte, figura, manifest_clima, archivos, inicio, fin, duracion):
    manifest_path = OUTPUT_DIR / NOMBRES_SALIDA['manifest']
    if manifest_path.exists() and not SOBRESCRIBIR_RESULTADOS:
        existente = json.loads(manifest_path.read_text(encoding='utf-8'))
        if existente.get('estado') == 'COMPLETA_SIN_POLIGONOS':
            print(f'La auditoría geográfica ya está completa; no se sobrescribe: {OUTPUT_DIR}')
            return existente
    if OUTPUT_DIR.exists() and any(OUTPUT_DIR.iterdir()) and not SOBRESCRIBIR_RESULTADOS:
        raise RuntimeError(f'Existe una salida geográfica incompleta: {OUTPUT_DIR}')

    escribir_json_atomico({
        'estado': 'INICIADA',
        'geography_version': GEOGRAPHY_VERSION,
        'inicio': inicio.isoformat(),
        'commit': detectar_commit(REPO_DIR),
    }, manifest_path, sobrescribir=SOBRESCRIBIR_RESULTADOS)

    tablas = {
        'catalogo': resultado.catalogo_climatico,
        'candidatos': resultado.estaciones_candidatas,
        'revision': resultado.estaciones_revision,
        'divipola': resultado.divipola_objetivo,
        'resumen': resultado.resumen,
    }
    salidas = {}
    for nombre, tabla in tablas.items():
        ruta = OUTPUT_DIR / NOMBRES_SALIDA[nombre]
        escribir_parquet_atomico(tabla, ruta, sobrescribir=SOBRESCRIBIR_RESULTADOS)
        salidas[nombre] = {'ruta': str(ruta), 'filas': len(tabla), 'bytes': ruta.stat().st_size}

    reporte_path = OUTPUT_DIR / NOMBRES_SALIDA['reporte']
    escribir_texto_atomico(reporte, reporte_path, sobrescribir=SOBRESCRIBIR_RESULTADOS)
    mapa_path = None
    if figura is not None:
        mapa_path = OUTPUT_DIR / NOMBRES_SALIDA['mapa']
        escribir_texto_atomico(
            figura.to_html(full_html=True, include_plotlyjs='cdn'),
            mapa_path,
            sobrescribir=SOBRESCRIBIR_RESULTADOS,
        )
    manifest = {
        'estado': resultado.metricas['estado'],
        'geography_version': GEOGRAPHY_VERSION,
        'commit': detectar_commit(REPO_DIR),
        'inicio': inicio.isoformat(),
        'fin': fin.isoformat(),
        'duracion_segundos': round(duracion, 2),
        'fuente_compartida_solo_lectura': str(SHARED_SOURCE_ROOT),
        'entrada_clima': {
            'ruta': str(CLIMATE_INPUT_DIR),
            'commit': manifest_clima.get('commit'),
            'regla_version': manifest_clima.get('regla_version'),
            'particiones': len(archivos),
        },
        'parametros': {
            'umbral_coordenadas_grados': UMBRAL_COORDENADAS_GRADOS,
            'validacion_poligonos': False,
        },
        'metricas': resultado.metricas,
        'salidas': salidas,
        'reporte': str(reporte_path),
        'mapa': str(mapa_path) if mapa_path is not None else None,
    }
    escribir_json_atomico(manifest, manifest_path, sobrescribir=True)
    print(f'Auditoría geográfica guardada en: {OUTPUT_DIR}')
    return manifest

## 4. Ejecución protegida

Primero confirme el plan con la bandera en `False`. Para ejecutar, cambie únicamente `EJECUTAR_AUDITORIA_GEOGRAFICA=True` y vuelva a correr desde la configuración.

In [ ]:
resultado_geografia = None

if not EJECUTAR_AUDITORIA_GEOGRAFICA:
    print('Auditoría geográfica desactivada. Revise el plan y active la bandera.')
else:
    inicio = ahora_proyecto()
    reloj = time.perf_counter()
    diario, estaciones, divipola, manifest_clima, archivos = cargar_entradas_geograficas()
    resultado_geografia = auditar_geografia(
        diario,
        estaciones,
        divipola,
        umbral_coordenadas_grados=UMBRAL_COORDENADAS_GRADOS,
    )
    figura_mapa = construir_mapa(resultado_geografia)
    fin = ahora_proyecto()
    duracion = time.perf_counter() - reloj
    reporte = construir_reporte(
        resultado_geografia,
        manifest_clima,
        inicio,
        fin,
        duracion,
    )

    display(Markdown('## Métricas geográficas'))
    display(pd.DataFrame([resultado_geografia.metricas]))
    display(Markdown('## Resumen por departamento'))
    display(resultado_geografia.resumen)
    display(Markdown('## Estaciones para revisión'))
    display(resultado_geografia.estaciones_revision[[
        'departamento', 'codigoestacion', 'Nombre', 'Municipio',
        'municipios_reportados', 'codigo_municipio',
        'motivos_revision_geografica',
    ]])
    if figura_mapa is not None:
        figura_mapa.show()
    print(f'Duración: {formatear_duracion(duracion)}')

    if GUARDAR_RESULTADOS:
        guardar_auditoria_geografica(
            resultado_geografia,
            reporte,
            figura_mapa,
            manifest_clima,
            archivos,
            inicio,
            fin,
            duracion,
        )
    else:
        print('Resultados no guardados porque GUARDAR_RESULTADOS=False.')

## 5. Compuerta antes de 07

La auditoría de puntos no habilita todavía la agregación municipal. Antes de 07 se debe:

1. Conseguir una capa completa de polígonos municipales con atributos y CRS.
2. Ejecutar punto-en-polígono para las 126 estaciones.
3. Resolver estaciones fuera del alcance, municipios múltiples y coordenadas discrepantes.
4. Publicar `estaciones_municipio.parquet` con asignación canónica, método y evidencia.
5. Aprobar las reglas específicas de agregación municipal por variable.